In [1]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [18]:
dias = ['domingo', 'segunda', 'terça', 'quarta', 'quinta', 'sexta', 'sábado']
dia_num = [0,1,2,3,4,5,6]
func = [10,7,5,5,6,7,8]

salario = [100,50,50,50,50,50,100]



In [20]:
model = pyo.ConcreteModel()

model.dias = pyo.Set(initialize=dia_num)
model.func = pyo.Param(model.dias, initialize=dict(zip(dia_num, func)))
model.salario = pyo.Param(model.dias, initialize=dict(zip(dia_num, salario)))
model.x = pyo.Var(model.dias, domain=pyo.NonNegativeIntegers)


In [21]:
#objetivo

def objetivo(model):
    return sum(model.salario[dia] * model.x[dia] for dia in model.dias)

model.obj = pyo.Objective(rule=objetivo, sense=pyo.minimize)

In [24]:
#restrição de demanda
#numero minimo de funcionarios por dia
def demanda(model, dia):
    return sum(model.x[(dia-k)%7] for k in range(6)) >= model.func[dia]

model.demanda = pyo.Constraint(model.dias, rule=demanda)

'pyomo.core.base.constraint.IndexedConstraint'>) on block unknown with a new
Component (type=<class 'pyomo.core.base.constraint.IndexedConstraint'>). This
is usually indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().


In [25]:
model.pprint()

1 Set Declarations
    dias : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    7 : {0, 1, 2, 3, 4, 5, 6}

2 Param Declarations
    func : Size=7, Index=dias, Domain=Any, Default=None, Mutable=False
        Key : Value
          0 :    10
          1 :     7
          2 :     5
          3 :     5
          4 :     6
          5 :     7
          6 :     8
    salario : Size=7, Index=dias, Domain=Any, Default=None, Mutable=False
        Key : Value
          0 :   100
          1 :    50
          2 :    50
          3 :    50
          4 :    50
          5 :    50
          6 :   100

1 Var Declarations
    x : Size=7, Index=dias
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          0 :     0 :  None :  None : False :  True : NonNegativeIntegers
          1 :     0 :  None :  None : False :  True : NonNegativeIntegers
          2 :     0 :  None :  None : False :  True : NonNegativeIntegers
       

In [26]:
res = SolverFactory('cplex')
opt = res.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmpld2dgc1r.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmp9i4ug6x2.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmp9i4ug6x2.pyomo.lp
Objective sense      : Minimize
Variables            :       7  [General Integer: 7]
Objective nonzeros   :       7
Linear constraints   :       7  [Greater: 7]
  Nonzeros           :      42
  RHS nonzeros       :       7

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 50.00000    

In [30]:
for re in model.x:
    print(f'funcionarios no dia: {dias[re]}: {model.x[re].value}')

funcionarios no dia: domingo: -0.0
funcionarios no dia: segunda: -0.0
funcionarios no dia: terça: 3.0
funcionarios no dia: quarta: 5.0
funcionarios no dia: quinta: 2.0
funcionarios no dia: sexta: -0.0
funcionarios no dia: sábado: -0.0
